In [51]:
import numpy as np
import random
import micrograd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [58]:
def run_model(act_layer):
    torch.manual_seed(7)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        act_layer,
        nn.Linear(4, 1)
    ).double()
    return model

class Exp(nn.Module):
    def forward(self, x):
        return torch.exp(x)

# Input and target
x = torch.tensor([[1.2, -0.8]]).double()
target = torch.tensor([[1.0]]).double()

# Create models
tanh = run_model(nn.Tanh())
exp_model = run_model(Exp())
def get_gradients(model):
    loss = nn.MSELoss()(model(x), target)
    loss.backward()

    linear_layers = [
        m for m in model if isinstance(m, nn.Linear)
    ]

    w_grads = [m.weight.grad.detach().numpy() for m in linear_layers]
    b_grads = [m.bias.grad.detach().numpy() for m in linear_layers]

    return w_grads, b_grads

tanh_w, tanh_b = get_gradients(tanh)
exp_w, exp_b = get_gradients(exp_model)

print("Tanh:")
print(tanh_w)
print(tanh_b)

print("Exp:")
print(exp_w)
print(exp_b)

Tanh:
[array([[ 1.40605834, -0.93737223],
       [ 1.15852083, -0.77234722],
       [ 0.        , -0.        ],
       [ 0.        , -0.        ]]), array([[-0.93244894, -2.501766  ,  0.        ,  0.        ]])]
[array([1.17171528, 0.96543402, 0.        , 0.        ]), array([-3.7548812])]
Exp:
[array([[ 3.06386264, -2.04257509],
       [ 7.38602077, -4.92401385],
       [ 0.        , -0.        ],
       [ 0.        , -0.        ]]), array([[ -7.67748221, -13.31199494,  -5.95755157,  -5.95755157]])]
[array([2.55321886, 6.15501731, 0.        , 0.        ]), array([-5.95755157])]


In [59]:
from micrograd.nn import MLP, Neuron
import math
from micrograd.engine import Value

def tanh(self):
  x = self.data
  t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
  out = Value(t, (self,), "tanh")

  def _backward():
    self.grad += (1 - t**2) * out.grad

  out._backward = _backward
  return out


def exp(self):
  x = self.data
  out = Value(math.exp(x), (self,), "exp")

  def _backward():
    self.grad += out.data * out.grad  # derivative of e^x is e^x

  out._backward = _backward
  return out


def log(self):
  x = self.data
  out = Value(math.log(x), (self,), "log")

  def _backward():
    self.grad += (1 / x) * out.grad  # derivative of ln(x) is 1/x

  out._backward = _backward
  return out

Value.tanh = tanh
Value.exp = exp
Value.log = log

Neuron.__call__ = lambda self, x: (
    lambda act: act.tanh() if self.nonlin else act
)(sum((wi * xi for wi, xi in zip(self.w, x)), self.b))


model = MLP(2, [4, 1])
x = [2.0, -3.0]


def forward():
    out = model(x)
    return (out.exp() + Value(1.0)).log()
    print("Forward output:", out)
    
out = forward()
model.zero_grad()
out.backward()

h = 1e-6
for i, p in enumerate(model.parameters()):
  p.label = f"p{i:02d}"
  orig = p.data

  p.data = orig + h
  f_plus = forward().data

  p.data = orig - h
  f_minus = forward().data

  p.data = orig  # restore original parameter value

  num_grad = (f_plus - f_minus) / (2 * h)
  diff = abs(p.grad - num_grad)
  status = "PASS" if diff < 1e-7 else "FAIL"


for i, p in enumerate(model.parameters()):
  p.label = f"p{i:02d}"

  # Central finite difference (CFD)
  orig = p.data
  p.data = orig + h
  f_plus = forward().data
  p.data = orig - h
  f_minus = forward().data
  p.data = orig
  cfd = (f_plus - f_minus) / (2 * h)

  # Safe log formatting for negative/zero parameters
  log_str = f"{p.log().data:+.4f}" if p.data > 0 else "   N/A "

  print(
      f"param {i:02d} | data: {p.data:+.4f} | grad: {p.grad:+.6f} | "
      f"cfd: {cfd:+.6f} | exp: {p.exp().data:+.4f} | tanh: {p.tanh().data:+.4f} |"
      f" log: {log_str}"
  )

param 00 | data: -0.5948 | grad: -0.003646 | cfd: -0.003646 | exp: +0.5517 | tanh: -0.5334 | log:    N/A 
param 01 | data: +0.8069 | grad: +0.005469 | cfd: +0.005469 | exp: +2.2409 | tanh: +0.6679 | log: -0.2146
param 02 | data: +0.0000 | grad: -0.001823 | cfd: -0.001823 | exp: +1.0000 | tanh: +0.0000 | log:    N/A 
param 03 | data: -0.0088 | grad: -0.062774 | cfd: -0.062774 | exp: +0.9913 | tanh: -0.0088 | log:    N/A 
param 04 | data: +0.6850 | grad: +0.094161 | cfd: +0.094161 | exp: +1.9837 | tanh: +0.5947 | log: -0.3784
param 05 | data: +0.0000 | grad: -0.031387 | cfd: -0.031387 | exp: +1.0000 | tanh: +0.0000 | log:    N/A 
param 06 | data: -0.7130 | grad: +0.280315 | cfd: +0.280315 | exp: +0.4902 | tanh: -0.6125 | log:    N/A 
param 07 | data: -0.9987 | grad: -0.420473 | cfd: -0.420473 | exp: +0.3684 | tanh: -0.7610 | log:    N/A 
param 08 | data: +0.0000 | grad: +0.140158 | cfd: +0.140158 | exp: +1.0000 | tanh: +0.0000 | log:    N/A 
param 09 | data: -0.7617 | grad: +0.466535 | c